# DSC 2026 Task 1 — Submit cấu hình validation tốt nhất (0.9524)

Notebook chỉ inference. Trước khi chạy: **Add Input → Your Work** và attach Output của
`dsc2026-task1-v3b-finetune-reranker` có `checkpoint-30` và embedding.

Chạy tuần tự tất cả cell. Notebook không fine-tune và không sweep validation.


In [ ]:
#@title Config
import os, sys, glob

# ---- Môi trường: Colab / Kaggle / local -------------------------------------
# Kiểm tra Kaggle TRƯỚC vì Kaggle không có /content, còn Colab thì luôn có.
# Trên Windows '/content' được hiểu là 'C:\content' nên phải chặn bằng os.name.
def _detect_env():
    if os.environ.get('KAGGLE_KERNEL_RUN_TYPE') or os.environ.get('KAGGLE_URL_BASE'):
        return 'kaggle'
    if os.name == 'posix' and os.path.isdir('/kaggle/input'):
        return 'kaggle'
    if 'google.colab' in sys.modules or os.environ.get('COLAB_RELEASE_TAG'):
        return 'colab'
    if os.name == 'posix' and os.path.isdir('/content'):
        return 'colab'
    return 'local'

ENV       = _detect_env()
ON_COLAB  = ENV == 'colab'
ON_KAGGLE = ENV == 'kaggle'
PROFILE   = 'v3b_ft_reranker'   # cache + tên file nộp riêng cho bản này.
# CHẠY SONG SONG HAI BẢN: trên Kaggle mỗi notebook có /kaggle/working
# riêng nên an toàn. Trên Colab thì KHÔNG — cả hai mount cùng My Drive;
# phải để PROFILE khác nhau VÀ cap khác nhau (xem _CAPTAG dưới), nếu
# không hai session ghi đè shard .npy của nhau và làm hỏng cả hai.

# ---- GPU ---------------------------------------------------------------------
# '' = dùng hết GPU thấy được (Kaggle: 2×T4 -> Stage 5 và 7 chia việc cho cả hai).
#      Đây là giá trị nên dùng: Stage 8 tự ép mình về 1 GPU trong lúc train (xem
#      cell đó), nên không còn phải hy sinh tốc độ encode để fine-tune được nữa.
# '0' = buộc 1 GPU cho toàn bộ notebook (chỉ cần khi debug).
# Đổi biến này BẮT BUỘC phải Restart runtime rồi chạy lại từ đầu (torch đọc
# CUDA_VISIBLE_DEVICES một lần duy nhất lúc khởi tạo).
VISIBLE_GPUS = ''   #@param ['', '0', '1', '0,1'] {allow-input: true}
if VISIBLE_GPUS:
    os.environ['CUDA_VISIBLE_DEVICES'] = VISIBLE_GPUS

# ---- Nơi GHI artifact --------------------------------------------------------
# Colab : My Drive CỦA BẠN (không ghi vào thư mục chia sẻ của nhóm: ~1.5GB và
#         người khác có thể xoá mất).
# Kaggle: /kaggle/working — 20GB, và là thư mục DUY NHẤT còn lại sau "Save Version".
#         /kaggle/temp không vào output nên dùng làm đĩa tạm.
if ON_COLAB:
    ART_ROOT = '/content/drive/MyDrive/DSC2026_Task1'
    SCRATCH  = '/content/scratch'
elif ON_KAGGLE:
    ART_ROOT = '/kaggle/working/DSC2026_Task1'
    SCRATCH  = '/kaggle/temp/scratch'
else:
    ART_ROOT = './work'
    SCRATCH  = './scratch'
ART = f'{ART_ROOT}/artifacts'

# ---- Cache CHỈ ĐỌC từ session trước (Kaggle) ---------------------------------
# Kaggle không có Drive: cách tái dùng embedding giữa các session là "Save Version"
# rồi attach output đó lại làm Dataset ở lần sau (xem Stage 11 ở cuối notebook).
#
# DÒ chứ không ghim pattern. Kaggle mount dataset ở độ sâu khác nhau tuỳ cách
# attach: /kaggle/input/<slug>/... khi attach từ "Your Work", nhưng
# /kaggle/input/datasets/<user>/<slug>/... khi attach dataset của người khác.
# Cộng thêm tiền tố DSC2026_Task1/ nằm sẵn trong zip là thành 4 tầng — pattern
# cứng trượt mà KHÔNG báo lỗi, notebook chỉ lặng lẽ encode lại toàn corpus.
# Nên: quét /kaggle/input tìm mọi thư mục tên 'artifacts', và chỉ nhận cái nào
# thật sự chứa artifact của pipeline này.
def _find_art_in(root='/kaggle/input', max_depth=6):
    found = []
    if not os.path.isdir(root):
        return found
    base = root.rstrip('/').count('/')
    for dirpath, dirnames, _ in os.walk(root):
        if dirpath.count('/') - base >= max_depth:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames
                       if not d.startswith('.') and d != '__MACOSX']
        if 'artifacts' in dirnames:
            p = os.path.join(dirpath, 'artifacts')
            # phải có dấu vết của pipeline, tránh nhận nhầm thư mục trùng tên
            if os.path.exists(f'{p}/corpus.parquet') or os.path.isdir(f'{p}/emb'):
                found.append(p)
    return sorted(dict.fromkeys(found))

ART_IN = _find_art_in() if ON_KAGGLE else []

def cache_path(rel):
    """Tìm cache theo thứ tự: ART (ghi được) -> ART_IN (chỉ đọc). None nếu chưa có."""
    p = f'{ART}/{rel}'
    if os.path.exists(p):
        return p
    for d in ART_IN:
        q = f'{d}/{rel}'
        if os.path.exists(q):
            return q
    return None

def first_existing(paths):
    for p in paths:
        if p and os.path.exists(p):
            return p
    return None

# ---- Dataset đã attach trên Kaggle -------------------------------------------
# Kaggle mount dataset theo /kaggle/input/<slug>, nhưng khi attach kèm tên chủ sở
# hữu thì đường dẫn thành /kaggle/input/datasets/<user>/<slug>. Ghim cả hai dạng;
# không trúng dạng nào thì cell sau tự dò trong /kaggle/input.
KAGGLE_DATA_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-publictest/LegalR - Public Test',
    '/kaggle/input/uit-task1-publictest/LegalR - Public Test',
]
KAGGLE_SCORING_HINTS = [
    '/kaggle/input/datasets/kitnehi1211/uit-task1-scoring/Scoring-Program-Task-LegalIR',
    '/kaggle/input/uit-task1-scoring/Scoring-Program-Task-LegalIR',
]
# scoring.py THẬT của BTC -> Stage 3 dùng để đối chiếu hàm chấm điểm tự viết
SCORING_DIR = first_existing(KAGGLE_SCORING_HINTS + [
    './Scoring-Program-Task-LegalIR', '../Scoring-Program-Task-LegalIR'])

# ---- Nơi ĐỌC dữ liệu: đường dẫn cố định theo môi trường ---------------------
# Đường dẫn nào không tồn tại thì cell sau tự dò.
DATA_FILES = ('train.json', 'public-official.json')
if ON_COLAB:
    RAW = '/content/drive/MyDrive/UIT DS Challenge/Task 1/LegalR - Public Test'
elif ON_KAGGLE:
    RAW = first_existing(KAGGLE_DATA_HINTS)
else:
    RAW = './LegalR - Public Test'
CTX_DIR = f'{SCRATCH}/contexts'   # nơi giải nén zip (đĩa tạm, KHÔNG vào Drive/output)

# ---- Chunking ----------------------------------------------------------------
CHUNK_CHARS        = 3000   # phù hợp context dài của Vietnamese_Embedding_v2
CHUNK_OVERLAP      = 500
MAX_CHUNKS_PER_DOC = 2500
# Đo trên dữ liệu thật: cap=150 cắt mất nội dung của 6.3% số lần gold doc
# xuất hiện, trong khi chỉ 41/8532 doc bị cap và bỏ cap chỉ thêm 3.5%
# chunk (154k -> 160k). Gold doc dài gấp ~3 lần trung bình corpus
# (p50 64k vs 22k ký tự) nên đây là mất mát thật.
#
# Cap PHẢI nằm trong tên cache. TAG/CHUNKS_REL bản v2 chỉ chứa
# CHUNK_CHARS, nên đổi cap mà không đổi tên là cache_path() trả về
# parquet + shard embedding CŨ trong im lặng. Giữ nguyên tên khi cap=150
# để cache của các lần chạy trước vẫn dùng lại được.
CHUNKER_VER = 't1'
# Đổi khi logic chunk_doc() ở Stage 2 đổi -> tên cache đổi theo. Thiếu cái này thì
# sửa chunker mà cache_path() vẫn trả về parquet + shard của chunker CŨ, im lặng.
_CAPTAG = (('' if MAX_CHUNKS_PER_DOC == 150 else f'_m{MAX_CHUNKS_PER_DOC}')
           + (f'_{CHUNKER_VER}' if CHUNKER_VER else ''))

# ---- Retrieval ---------------------------------------------------------------
BM25_K1, BM25_B = 0.9, 0.4  # tuned cho document dài
CAND_CHUNKS     = 1000      # số chunk lấy ra trước khi gộp về document
CAND_DOCS       = 100       # số doc mỗi nhánh đưa vào RRF
RRF_K           = 60
RERANK_TOP      = 30       # số doc đưa vào cross-encoder
TOPK_SUBMIT     = 5         # RÀNG BUỘC CỨNG của BTC

# Cách dùng điểm của cross-encoder. ĐỪNG đoán — Stage 7 in cả ba, chọn theo val:
#   'rrf'     : RRF giữa thứ tự hybrid và thứ tự rerank. Reranker sai một câu thì
#               phiếu của retriever vẫn giữ được gold trong top-5 -> an toàn nhất.
#   'replace' : tin hẳn reranker (cách cũ). Chỉ chọn khi val cho thấy nó thắng rõ.
#   'none'    : bỏ rerank. Đúng là một lựa chọn: ở lần chạy thử trên corpus rút
#               gọn, hybrid RRF (recall@5 0.919) còn cao hơn 'replace' (0.858).
RERANK_MODE     = 'rrf'   #@param ['rrf', 'replace', 'none']

# ---- Ba tham số thêm sau khi đọc log profile v2 ------------------------------
# Log v2 nói rất rõ:
#   dense              recall@5 = 0.8996  <- TỐT NHẤT
#   hybrid RRF         recall@5 = 0.8699  <- tụt 3 điểm so với dense
#   hybrid + rerank    recall@5 = 0.8973  <- vẫn chưa vượt nổi dense đơn thuần
# Nguyên nhân: RRF cho BM25 (recall@5 = 0.6868) một phiếu NGANG BẰNG dense.
# Và dense recall@100 = 0.9897 -> khâu tìm ứng viên hết chỗ cải thiện; toàn bộ
# dư địa còn lại nằm ở XẾP HẠNG trong top-30 (trần ~0.975).

# Base đưa vào cross-encoder và dùng cho submission.
# Stage 7 đo CẢ HAI rồi in ra cái nào thắng — sửa biến này theo bảng đó.
RERANK_BASE = 'dense'   #@param ['dense', 'hybrid']

# Trọng số RRF cho (bm25, dense). (1,1) là RRF cổ điển, chỉ đúng khi hai kênh
# mạnh ngang nhau. Stage 6 sweep vài giá trị rồi chọn theo val.
RRF_WEIGHTS = (1.0, 3.0)

# Số chunk mỗi doc đưa cho cross-encoder; điểm doc = MAX trên các chunk đó.
# Trước đây = 1: với nhiều chunk/doc, reranker chỉ
# được đọc 1/150 nội dung của doc -> chọn nhầm chunk đại diện là gold chết oan.
# Chi phí GPU tăng tuyến tính (30 doc x 3 chunk = 90 cặp/query).
RERANK_CHUNKS_PER_DOC = 8   # bản B: cố định = config lúc train reranker

# ---- Models (phải đăng ký với BTC trước khi dùng) ----------------------------
BIENCODER = 'AITeamVN/Vietnamese_Embedding_v2'    # 0.6B
RERANKER  = 'AITeamVN/Vietnamese_Reranker'        # 0.6B -> tổng 1.2B < 4B
MAX_SEQ   = 1024
def format_dense_queries(texts):
    return list(texts)
def format_dense_passages(texts):
    return list(texts)
FT_BIENCODER_PATH = f'{ART}/models/biencoder-ft-{PROFILE}'   # Stage 8 ghi ra đây
# Stage 5 encode bằng checkpoint nào lúc BẮT ĐẦU chạy notebook.
#   False -> checkpoint gốc. Đây là giá trị đúng cho lần chạy đầu: Stage 8 cần
#            retrieval của model gốc để mine hard negative.
#   Stage 8 train xong sẽ TỰ bật biến này lên True + encode lại toàn corpus,
#   nên không cần sửa tay rồi chạy lại notebook từ đầu nữa.
USE_FINETUNED = False
DO_FINETUNE   = True   #@param {type:'boolean'} — Stage 8 (bi-encoder)
# TẮT ở cả hai bản v3. recall@100 = 0.9897: retriever đã xong việc, dư địa
# nằm ở XẾP HẠNG. Fine-tune bi-encoder tốn thêm một lần encode lại toàn
# corpus mà tối ưu đúng cái đã hết chỗ tối ưu.

# ---- Eval --------------------------------------------------------------------
N_VAL = 1000
SEED  = 42

# ---- Batch: các số này là batch cho MỖI GPU ---------------------------------
# T4 Colab 16GB và T4 Kaggle 15GB gần như bằng nhau -> giữ nguyên batch, tốc độ
# tăng là do chia việc cho 2 GPU chứ không phải do batch to hơn. Kaggle P100
# (1×16GB) cũng chạy đúng cấu hình này.
ENC_SHARD     = 20000   # encode theo shard, checkpoint từng shard ra đĩa
ENC_BATCH     = 16
RERANK_MAXLEN = 512      # bản B: passage ngắn tập trung, = maxlen lúc train
RERANK_BATCH  = 8
RERANK_FP16   = True     # reranker chạy fp16: ~2x nhanh, chênh lệch điểm không đáng kể

print('ENV =', ENV, '| PROFILE =', PROFILE, '| RERANK_MODE =', RERANK_MODE)
print('ART      =', ART)
print('SCRATCH  =', SCRATCH)
if RAW:
    print('RAW      =', RAW)
if SCORING_DIR:
    print('scoring.py BTC:', SCORING_DIR)
if ART_IN:
    print('cache chỉ đọc:', ART_IN)


In [ ]:
#@title Ghim artifact từ dataset testing
TESTING_ART = '/kaggle/input/datasets/kitnehi1211/testing/DSC2026_Task1/artifacts'
assert os.path.isdir(TESTING_ART), f'Không tìm thấy artifact dataset: {TESTING_ART}'
assert os.path.isdir(f'{TESTING_ART}/emb'), f'Không tìm thấy embedding: {TESTING_ART}/emb'
assert os.path.isfile(
    f'{TESTING_ART}/models/reranker-ft-v3b_ft_reranker/checkpoint-30/model.safetensors'
), 'Dataset testing không có checkpoint-30/model.safetensors'

# Chỉ đọc cache từ đúng version đã đóng gói vào dataset testing.
ART_IN = [TESTING_ART]
print('cache chỉ đọc đã ghim:', ART_IN)


In [ ]:
#@title Mount Drive (Colab) + cài dependency + liệt kê GPU
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# Pin cùng một bản Sentence Transformers trên cả Colab lẫn Kaggle.
ST_WANT = '5.4.0'  # tương thích cả legal E5 và Vietnamese cross-encoder
try:
    import sentence_transformers as _st
    ST_HAVE = _st.__version__
except ImportError:
    ST_HAVE = None

if ST_HAVE != ST_WANT:
    !pip -q install sentence-transformers=={ST_WANT} pyarrow 2>&1 | tail -1
    import importlib, sentence_transformers as _st
    ST_HAVE = importlib.reload(_st).__version__

print('sentence-transformers', ST_HAVE, '(bản đã test:', ST_WANT + ')')

import os
for d in (ART, f'{ART}/emb', f'{ART}/models', SCRATCH):
    os.makedirs(d, exist_ok=True)

import torch
N_GPU   = torch.cuda.device_count()
DEVICES = [f'cuda:{i}' for i in range(N_GPU)] or ['cpu']
DEV     = DEVICES[0]

print('torch', torch.__version__, '| GPU thấy được:', N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  cuda:{i}  {p.name}  {p.total_memory/1e9:.1f} GB')
if N_GPU > 1:
    print(f'-> Stage 5 (encode corpus) và Stage 7 (rerank) sẽ chia việc cho {N_GPU} GPU')
    print('   Stage 8 (fine-tune) tự ép về 1 GPU trong lúc train, không cần sửa gì.')
elif N_GPU == 1:
    print('-> 1 GPU: pipeline chạy tuần tự như bình thường')
else:
    print('-> KHÔNG có GPU. Stage 1-4 (BM25) vẫn chạy được và nên chạy ở session CPU '
          'để tiết kiệm quota; Stage 5 trở đi bắt buộc phải có GPU.')


In [ ]:
#@title Dò dữ liệu + giải nén selected-contexts.zip
# Ưu tiên RAW cố định ở Config; không có thì dò (Kaggle: /kaggle/input/<dataset>).
# Giải nén vào đĩa tạm chứ KHÔNG vào Drive / /kaggle/working: 8.532 file nhỏ ghi
# lên Drive mất hàng giờ, và đếm vào giới hạn 20GB output của Kaggle.
import os, glob, zipfile, time

def find_data_dir(roots, max_depth=6):
    """Thư mục đầu tiên chứa đủ DATA_FILES."""
    for root in roots:
        if not os.path.isdir(root):
            continue
        base = root.rstrip('/').count('/')
        for dirpath, dirnames, filenames in os.walk(root):
            if dirpath.count('/') - base >= max_depth:
                dirnames[:] = []
                continue
            dirnames[:] = [d for d in dirnames
                           if not d.startswith('.') and d != '__MACOSX']
            if all(f in filenames for f in DATA_FILES):
                return dirpath
    return None

def data_roots():
    if ON_COLAB:
        return ['/content/drive/MyDrive',
                '/content/drive/.shortcut-targets-by-id',   # nơi shortcut trỏ tới
                '/content/drive/Shareddrives',
                '/content']
    if ON_KAGGLE:
        return ['/kaggle/input']                            # mọi dataset đã attach
    return ['.', '..']

def n_ctx(d):
    return len(glob.glob(f'{d}/**/context_*.json', recursive=True)) if d else 0

if RAW is None or not all(os.path.exists(f'{RAW}/{f}') for f in DATA_FILES):
    print('đường dẫn cố định không dùng được -> đang dò...', flush=True)
    RAW = find_data_dir(data_roots())

assert RAW, (
    'KHÔNG tìm thấy dữ liệu (cần train.json + public-official.json).\n\n'
    'Colab: thư mục "UIT DS Challenge" thường ở "Shared with me" mà Colab không\n'
    '  mount phần đó. Sửa 1 lần: drive.google.com -> Shared with me -> chuột phải\n'
    '  "UIT DS Challenge" -> Organise -> Add shortcut to Drive -> My Drive.\n\n'
    'Kaggle: phải Add Input dataset chứa dữ liệu Task 1 vào notebook\n'
    '  (Notebook -> Add Input -> Datasets). Kiểm tra: !ls /kaggle/input\n\n'
    'Hoặc gán tay: RAW = "<đường dẫn tới thư mục LegalR - Public Test>"')

print('RAW =', RAW)
CTX_ZIP = f'{RAW}/selected-contexts.zip'

if cache_path('corpus.parquet'):
    print('corpus.parquet đã có -> bỏ qua giải nén')
elif n_ctx(RAW):
    CTX_DIR = RAW                     # dataset đã bung sẵn (hay gặp trên Kaggle)
    print(f'đã bung sẵn trong RAW: {n_ctx(CTX_DIR)} file')
elif n_ctx(CTX_DIR):
    print(f'đã giải nén trước đó: {n_ctx(CTX_DIR)} file')
else:
    assert os.path.exists(CTX_ZIP), (
        f'Không thấy {CTX_ZIP} và cũng không thấy file context_*.json nào.\n'
        'Colab: kiểm tra shortcut "UIT DS Challenge" trong My Drive.\n'
        'Kaggle: dataset đã attach phải chứa selected-contexts (zip hoặc đã bung).')
    os.makedirs(CTX_DIR, exist_ok=True)
    t0 = time.time()
    with zipfile.ZipFile(CTX_ZIP) as z:
        print(f'{len(z.namelist())} entry, đang giải nén...', flush=True)
        z.extractall(CTX_DIR)
    print(f'xong {time.time()-t0:.0f}s -> {n_ctx(CTX_DIR)} file context')

if not cache_path('corpus.parquet'):
    n = n_ctx(CTX_DIR)
    assert n == 8532, f'mong đợi 8532 file context, thấy {n} — zip/dataset có thể thiếu'
    print('OK: đủ 8532 văn bản')


In [ ]:
#@title Build corpus.parquet
import json, re, glob, os, time
import pandas as pd

CORPUS_PQ = f'{ART}/corpus.parquet'

def _title_from_link(link):
    slug = (link or '').rstrip('/').rsplit('/', 1)[-1]
    slug = re.sub(r'\.aspx$', '', slug, flags=re.I)
    return slug.replace('-', ' ').strip()

def build_corpus():
    # Chỉ đọc CTX_DIR — cell "Giải nén selected-contexts.zip" ở trên đã bung sẵn.
    # Zip lồng hai lớp (selected-contexts/selected-contexts/...) nên glob đệ quy.
    files = sorted(glob.glob(f'{CTX_DIR}/**/context_*.json', recursive=True))
    assert files, f'{CTX_DIR} trống — hãy chạy lại cell "Giải nén selected-contexts.zip".'
    print(f'{len(files)} file context', flush=True)

    rows, n_noname = [], 0
    for i, fp in enumerate(files):
        with open(fp, encoding='utf-8') as f:
            d = json.load(f)
        title = (d.get('name') or '').replace('-', ' ').strip()
        if not title:
            n_noname += 1
            title = _title_from_link(d.get('link', ''))
        # gộp mọi whitespace: passage có rất nhiều khoảng trắng / newline rác.
        # Làm sớm để regex tách 'Điều N' ở Stage 2 hoạt động ổn định.
        passage = re.sub(r'\s+', ' ', d.get('passage') or '').strip()
        rows.append({'doc_id': str(d['id']), 'title': title,
                     'link': d.get('link', ''), 'passage': passage})
        if (i + 1) % 2000 == 0:
            print(f'  {i+1}/{len(files)}', flush=True)

    df = pd.DataFrame(rows).drop_duplicates('doc_id').reset_index(drop=True)
    print(f'-> {len(df)} doc | thiếu name (đã vá từ link): {n_noname}')
    df.to_parquet(CORPUS_PQ, index=False)
    return df

t0 = time.time()
_pq = cache_path('corpus.parquet')      # ART trước, rồi tới dataset chỉ đọc (Kaggle)
if _pq:
    corpus = pd.read_parquet(_pq)
    print(f'load cache: {len(corpus)} doc <- {_pq}')
else:
    corpus = build_corpus()

print('%.1fs' % (time.time() - t0))
print('passage chars: mean %d | p50 %d | max %d' % (
    corpus.passage.str.len().mean(), corpus.passage.str.len().median(),
    corpus.passage.str.len().max()))
corpus.head(3)[['doc_id', 'title']]

In [ ]:
#@title Load train / public + val split cố định
import json, random

with open(f'{RAW}/train.json', encoding='utf-8') as f:
    train_raw = json.load(f)
with open(f'{RAW}/public-official.json', encoding='utf-8') as f:
    public_raw = json.load(f)

# answer có thể là int hoặc str tuỳ file -> luôn ép str
QUESTIONS = {qid: v['question'] for qid, v in train_raw.items()}
GOLD = {qid: set(map(str, v['answer'])) for qid, v in train_raw.items()}
PUBLIC = {qid: v['question'] for qid, v in public_raw.items()}

# Split cố định theo seed + sorted key -> mọi session cho cùng một val set.
# Đây là điều kiện bắt buộc để so sánh các thí nghiệm với nhau.
all_qids = sorted(train_raw.keys())
random.Random(SEED).shuffle(all_qids)
VAL_QIDS   = all_qids[:N_VAL]
TRAIN_QIDS = all_qids[N_VAL:]

print(f'train {len(TRAIN_QIDS)} | val {len(VAL_QIDS)} | public {len(PUBLIC)}')

# kiểm tra gold có nằm trong corpus không
corpus_ids = set(corpus.doc_id)
missing = {g for q in all_qids for g in GOLD[q]} - corpus_ids
print('gold doc thiếu trong corpus:', len(missing))

In [ ]:
#@title Build chunks.parquet
import re, os, time
import pandas as pd

CHUNKS_REL = f'chunks_c{CHUNK_CHARS}{_CAPTAG}.parquet'
CHUNKS_PQ  = f'{ART}/{CHUNKS_REL}'

DIEU_RE = re.compile(r'(?=Điều\s+\d+\s*[\.\:])')

def _split_long(s, size, overlap):
    """Cắt s thành các đoạn <= size ký tự, ưu tiên cắt ở ranh giới câu."""
    out, i = [], 0
    while i < len(s):
        j = min(i + size, len(s))
        if j < len(s):
            # lùi về dấu chấm gần nhất trong 200 ký tự cuối cho khỏi cắt giữa câu
            cut = s.rfind('. ', i + size - 200, j)
            if cut > i:
                j = cut + 1
        out.append(s[i:j].strip())
        if j >= len(s):
            break
        i = max(j - overlap, i + 1)
    return [c for c in out if c]

def chunk_doc(title, passage):
    """-> list các đoạn text (chưa gắn tiêu đề)."""
    pieces = [p.strip() for p in DIEU_RE.split(passage) if p.strip()]
    if not pieces:
        pieces = [passage]

    merged = []
    buf = ''
    for p in pieces:
        if len(p) > CHUNK_CHARS:
            if buf:
                merged.append(buf); buf = ''
            merged.extend(_split_long(p, CHUNK_CHARS, CHUNK_OVERLAP))
        elif len(buf) + len(p) + 1 <= CHUNK_CHARS:
            buf = f'{buf} {p}'.strip()
        else:
            merged.append(buf); buf = p
    if buf:
        merged.append(buf)

    # chunk mở đầu: chứa số hiệu / cơ quan ban hành / trích yếu
    head = passage[:CHUNK_CHARS].strip()
    if head and (not merged or merged[0][:200] != head[:200]):
        merged.insert(0, head)
    return merged[:MAX_CHUNKS_PER_DOC]

def build_chunks(corpus):
    rows, n_titleonly = [], 0
    for n, r in enumerate(corpus.itertuples(index=False)):
        cs = chunk_doc(r.title, r.passage)
        if not cs:
            # 20 văn bản có `passage` RỖNG và cũng không có field `name` (tiêu đề
            # được vá từ slug trong `link` ở trên). chunk_doc() chỉ chunk passage
            # nên chúng nhận 0 chunk -> dense retrieval KHÔNG BAO GIỜ truy hồi
            # được. 6/20 doc này từng là gold, và 9 câu hỏi train mất trắng vì mọi
            # gold của chúng nằm trong nhóm đó. Phát một chunk chỉ chứa tiêu đề:
            # tiêu đề vá từ slug link có loại văn bản + chủ đề + năm, đủ để khớp ở
            # mức chủ đề — hơn hẳn không có gì.
            cs, n_titleonly = [''], n_titleonly + 1
        for c in cs:
            # tiêu đề đi kèm mọi chunk -> chunk lẻ vẫn biết mình thuộc văn bản nào
            rows.append((r.doc_id, f'{r.title}. {c}'.strip()))
        if (n + 1) % 1000 == 0:
            print(f'  {n+1}/{len(corpus)} doc -> {len(rows)} chunk', flush=True)
    if n_titleonly:
        print(f'-> {n_titleonly} doc passage rỗng: đã phát chunk chỉ-tiêu-đề')
    df = pd.DataFrame(rows, columns=['doc_id', 'text'])
    df.to_parquet(CHUNKS_PQ, index=False)
    return df

t0 = time.time()
_cpq = cache_path(CHUNKS_REL)
if _cpq:
    chunks = pd.read_parquet(_cpq)
    print('load cache <-', _cpq)
else:
    chunks = build_chunks(corpus)
print('%d chunk | %.1fs' % (len(chunks), time.time() - t0))

# map chunk -> doc dạng số nguyên, dùng suốt phần sau
DOC_IDS = corpus.doc_id.tolist()
DOC2IX  = {d: i for i, d in enumerate(DOC_IDS)}
import numpy as np
CHUNK2DOC = np.array([DOC2IX[d] for d in chunks.doc_id], dtype=np.int32)
print('chunk/doc trung bình: %.1f' % (len(chunks) / len(corpus)))
chunks.text.str.len().describe()[['mean', '50%', 'max']]

In [ ]:
#@title Metrics + experiment log
import numpy as np, pandas as pd, os, datetime

def official_score(sub, truth):
    """Bản sao CHÍNH XÁC của scoring.py mà BTC dùng trên Codabench.

    sub: {qid: {'answer': [id]}} — đúng định dạng file nộp
    truth: {qid: [id]}

    Ba chi tiết trong code của BTC mà đọc mô tả không thấy:
    1. Recall KHÔNG cắt top-5. Nó chỉ *chặn* ở len<=5: quá 5 id -> câu đó 0 điểm.
       Nộp đúng 5 id là tối ưu cho Recall.
    2. Số lượng key phải khớp tuyệt đối, lệch là raise Exception -> TOÀN BỘ bài nộp
       lỗi, không phải chỉ mất điểm câu đó.
    3. Thiếu key -> len(None) -> TypeError -> cũng sập toàn bài.
    """
    y_pred = {k: v['answer'] for k, v in sub.items()}
    if len(y_pred) != len(truth):
        raise Exception('Samples in predict not match with reference')
    ok = lambda k: 0 < len(y_pred[k]) <= TOPK_SUBMIT
    recall = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(truth[k])
                      if ok(k) else 0 for k in truth])
    precision = np.mean([len(set(truth[k]) & set(y_pred[k])) / len(y_pred[k])
                         if ok(k) else 0 for k in y_pred])
    return {'recall': float(recall), 'precision': float(precision)}

def evaluate(preds, gold, ks=(5, 10, 20, 30, 50, 100)):
    """preds: {qid: [doc_id đã xếp hạng]} -> recall@k chẩn đoán + điểm chính thức.

    recall@20/@100 KHÔNG phải metric của cuộc thi — chúng để chẩn đoán:
      @100 thấp        -> retriever yếu (văn bản đúng không vào nổi danh sách)
      @100 cao, @5 thấp -> ranking yếu (tìm ra rồi mà không đẩy lên top)

    recall@RERANK_TOP (mặc định @30) là **trần cứng** của Stage 7: cross-encoder
    chỉ được nhìn top-RERANK_TOP, doc không lọt vào đó thì không cách nào cứu.
    Đọc `@30` và `@5` cạnh nhau -> biết ngay còn bao nhiêu điểm đang bỏ trên bàn.
    """
    out = {}
    for k in ks:
        out[f'recall@{k}'] = float(np.mean([
            len(gold[q] & set(preds.get(q, [])[:k])) / len(gold[q]) for q in gold]))
    # điểm chính thức: dựng đúng shape file nộp rồi chấm bằng code của BTC
    sub = {q: {'answer': list(preds.get(q, [])[:TOPK_SUBMIT])} for q in gold}
    off = official_score(sub, {q: list(gold[q]) for q in gold})
    out['OFFICIAL_recall'] = off['recall']
    out['OFFICIAL_precision'] = off['precision']
    return out

def show(name, preds, gold=None, qids=None):
    gold = gold or {q: GOLD[q] for q in (qids or VAL_QIDS)}
    m = evaluate(preds, gold)
    print(name.ljust(28), ' '.join(f'{k}={v:.4f}' for k, v in m.items()))
    return m

LOG_CSV = f'{ART}/experiments.csv'

def log_exp(name, metrics, note=''):
    """Ghi lại mọi thí nghiệm. Bảng này là thứ bạn cần để chọn cấu hình cho private
    test và để viết paper nếu vào top 10 — đừng bỏ qua."""
    row = {'time': datetime.datetime.now().isoformat(timespec='seconds'),
           'profile': PROFILE, 'run': name, 'note': note, **metrics}
    df = pd.DataFrame([row])
    if os.path.exists(LOG_CSV):
        df = pd.concat([pd.read_csv(LOG_CSV), df], ignore_index=True)
    df.to_csv(LOG_CSV, index=False)
    return df.tail(5)

print('ok')


# ---- Đối chiếu official_score() với scoring.py THẬT của BTC ------------------
# Dataset uit-task1-scoring có sẵn mã chấm điểm Codabench -> nạp trực tiếp
# eval_retrieval() của BTC và so với bản chép tay ở trên trên vài trường hợp
# biên. Nếu lệch thì mọi con số OFFICIAL_* trong notebook đều vô nghĩa.
import importlib.util, glob as _glob

def _load_btc_eval():
    cands = ([f'{SCORING_DIR}/scoring.py'] if SCORING_DIR else []) + \
            sorted(_glob.glob('/kaggle/input/**/Scoring-Program-Task-LegalIR/scoring.py',
                              recursive=True))
    for p in cands:
        if not os.path.exists(p):
            continue
        spec = importlib.util.spec_from_file_location('btc_scoring', p)
        mod = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(mod)          # main() có guard __main__ -> import an toàn
        return mod.eval_retrieval, p
    return None, None

_btc_eval, _btc_path = _load_btc_eval()
if _btc_eval is None:
    print('không thấy scoring.py của BTC -> bỏ qua đối chiếu '
          '(Kaggle: Add Input dataset uit-task1-scoring)')
else:
    _truth = {'a': ['1'], 'b': ['2', '3'], 'c': ['4']}
    _cases = {
        'đúng 1/1':        {'a': ['1'], 'b': ['2', '3'], 'c': ['4']},
        'nộp 5 id':        {'a': ['1', 'x', 'y', 'z', 'w'], 'b': ['2', '3', 'x', 'y', 'z'],
                            'c': ['q', 'w', 'e', 'r', 't']},
        'nộp 6 id -> 0':   {'a': ['1', 'x', 'y', 'z', 'w', 'v'], 'b': ['2'], 'c': ['4']},
    }
    _ok = True
    for _name, _p in _cases.items():
        _mine = official_score({k: {'answer': v} for k, v in _p.items()}, _truth)
        _theirs = _btc_eval({k: {'answer': v} for k, v in _p.items()}, _truth)
        _same = (abs(_mine['recall'] - _theirs['recall']) < 1e-9 and
                 abs(_mine['precision'] - _theirs['precision']) < 1e-9)
        _ok &= _same
        print(f'  {_name:16s} recall={_theirs["recall"]:.4f} '
              f'precision={_theirs["precision"]:.4f}  {"khớp" if _same else "LỆCH!"}')
    assert _ok, 'official_score() lệch so với scoring.py của BTC — sửa trước khi tin số liệu'
    print('official_score() khớp scoring.py của BTC <-', _btc_path)


In [ ]:
#@title Gộp chunk -> doc bằng MAX, rồi lấy top-N doc
def chunks_to_docs(chunk_scores, chunk_top_ix, topn=CAND_DOCS):
    """Điểm doc = MAX điểm các chunk của nó (không phải mean).

    Gold là "văn bản CHỨA thông tin cần thiết" -> một chunk khớp là đủ. Lấy mean sẽ
    phạt oan các văn bản dài mà chỉ một Điều liên quan.
    """
    best = {}
    for ix, s in zip(chunk_top_ix, chunk_scores):
        d = CHUNK2DOC[ix]
        if s > best.get(d, -1e30):
            best[d] = s
    ranked = sorted(best.items(), key=lambda x: -x[1])[:topn]
    return [DOC_IDS[d] for d, _ in ranked], [s for _, s in ranked]

print('ok')

In [ ]:
#@title Encode corpus (đa GPU, sharded, resume được)
import numpy as np, os, glob, time, re, torch
from sentence_transformers import SentenceTransformer

def set_dense_model(model_path):
    """Chọn checkpoint cho nhánh dense + trỏ EMB_DIR sang cache của đúng checkpoint đó.

    Tách thành hàm vì Stage 8 gọi lại nó sau khi fine-tune: model đổi thì embedding
    cũ vô giá trị, phải đổi cả nơi ghi lẫn nơi đọc chứ không chỉ đổi mỗi model.
    """
    global MODEL_PATH, TAG, EMB_REL, EMB_DIR, EMB_DIRS
    MODEL_PATH = model_path
    TAG     = re.sub(r'[^a-zA-Z0-9]+', '_', MODEL_PATH) + f'_c{CHUNK_CHARS}{_CAPTAG}'
    EMB_REL = f'emb/{TAG}'
    EMB_DIR = f'{ART}/{EMB_REL}'          # nơi GHI
    os.makedirs(EMB_DIR, exist_ok=True)
    # nơi ĐỌC: thư mục ghi + mọi cache chỉ đọc (Kaggle: output session trước attach lại)
    EMB_DIRS = [EMB_DIR] + [f'{d}/{EMB_REL}' for d in ART_IN if os.path.isdir(f'{d}/{EMB_REL}')]
    print('model    ->', MODEL_PATH)
    print('ghi      ->', EMB_DIR)
    if len(EMB_DIRS) > 1:
        print('đọc thêm <-', EMB_DIRS[1:])

def _shard_file(a):
    """shard đã encode ở BẤT KỲ cache nào -> path, chưa có -> None."""
    for d in EMB_DIRS:
        p = f'{d}/s_{a:08d}.npy'
        if os.path.exists(p):
            return p
    return None

def _l2(v):
    """normalize tay: encode_multi_process ở một số bản ST không nhận
    normalize_embeddings, làm ở đây thì mọi bản đều cho ra vector cùng thang."""
    v = np.asarray(v, dtype=np.float32)
    v /= np.linalg.norm(v, axis=1, keepdims=True) + 1e-12
    return v

def encode_corpus():
    texts  = format_dense_passages(chunks.text.tolist())
    shards = [(i, min(i + ENC_SHARD, len(texts))) for i in range(0, len(texts), ENC_SHARD)]
    todo   = [(a, b) for a, b in shards if _shard_file(a) is None]

    if todo:
        model = SentenceTransformer(MODEL_PATH, device=DEV)
        model.max_seq_length = MAX_SEQ
        if torch.cuda.is_available():
            model.half()

        # >1 GPU: mỗi GPU một process con, ST tự chia đều sentence cho các process.
        # Đây là chỗ ăn tốc độ lớn nhất của Kaggle 2×GPU — encode là bước nặng nhất.
        pool = None
        if len(DEVICES) > 1:
            try:
                pool = model.start_multi_process_pool(target_devices=DEVICES)
                print(f'multi-GPU pool: {DEVICES}', flush=True)
            except Exception as e:
                print('không mở được multi-process pool (%s) -> chạy 1 GPU' % e, flush=True)
                pool = None

        print(f'{len(todo)}/{len(shards)} shard cần encode', flush=True)
        try:
            for a, b in todo:
                t0 = time.time()
                if pool is not None:
                    v = model.encode_multi_process(
                        texts[a:b], pool, batch_size=ENC_BATCH,
                        chunk_size=max(500, (b - a) // (4 * len(DEVICES))))
                else:
                    v = model.encode(texts[a:b], batch_size=ENC_BATCH,
                                     convert_to_numpy=True, show_progress_bar=False)
                np.save(f'{EMB_DIR}/s_{a:08d}.npy', _l2(v).astype(np.float16))
                print(f'  shard {a}-{b}: {time.time()-t0:.0f}s', flush=True)
        finally:
            if pool is not None:
                model.stop_multi_process_pool(pool)
            del model
            torch.cuda.empty_cache()

    return np.concatenate([np.load(_shard_file(a)) for a, _ in shards])

set_dense_model(FT_BIENCODER_PATH if USE_FINETUNED else BIENCODER)
t0  = time.time()
EMB = encode_corpus()
assert len(EMB) == len(chunks), f'emb {len(EMB)} != chunks {len(chunks)} — xoá {EMB_DIR} và encode lại'
print('EMB', EMB.shape, EMB.dtype, '| %.1f MB | %.1fs' % (EMB.nbytes / 1e6, time.time() - t0))


In [ ]:
#@title Dense search trên GPU
import torch, numpy as np
from sentence_transformers import SentenceTransformer

# EMB fp16 chỉ vài trăm MB nên để trọn trên 1 GPU: q @ EMB.T là một phép matmul,
# chia cho 2 GPU chỉ thêm đồng bộ chứ không nhanh hơn đáng kể. GPU còn lại được
# dùng thật sự ở Stage 5 (encode) và Stage 7 (rerank) — hai bước nặng.
EMB_T  = None
qmodel = None

def free_dense():
    """Nhả EMB_T + query encoder khỏi VRAM. Stage 8 gọi trước khi train: chỗ này
    đang giữ vài GB, để nguyên thì fine-tune gần như chắc chắn OOM."""
    global EMB_T, qmodel
    EMB_T = qmodel = None
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def load_query_model():
    """Đưa EMB lên GPU + nạp encoder cho query theo MODEL_PATH hiện tại.
    Gọi lại được — Stage 8 dùng để chuyển sang model vừa fine-tune."""
    global EMB_T, qmodel
    free_dense()
    EMB_T = torch.from_numpy(EMB).to(DEV)                   # (N, dim) fp16
    qmodel = SentenceTransformer(MODEL_PATH, device=DEV)
    qmodel.max_seq_length = MAX_SEQ
    if DEV.startswith('cuda'):
        qmodel.half()

def dense_search(qtext, topk=CAND_CHUNKS, batch=32):
    """-> (indices, scores) của top chunk cho mỗi query."""
    ixs, scs = [], []
    for i in range(0, len(qtext), batch):
        q = qmodel.encode(format_dense_queries(qtext[i:i+batch]), batch_size=batch, normalize_embeddings=True,
                          convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        s = q @ EMB_T.T                                     # cosine (đã normalize)
        v, ix = torch.topk(s, min(topk, s.shape[1]), dim=1)
        ixs.append(ix.cpu().numpy()); scs.append(v.float().cpu().numpy())
    return np.concatenate(ixs), np.concatenate(scs)

def retrieve_dense(qids, qtext):
    ix, sc = dense_search(qtext)
    return {q: chunks_to_docs(s, i)[0] for q, i, s in zip(qids, ix, sc)}

load_query_model()
print('Dense model + embedding ready')


In [ ]:
#@title 7B.2 — select_chunks: top-m chunk TRONG từng doc (thay best_chunks_per_doc)
# Lỗi của bản cũ: nó chỉ xét chunk nằm trong top-CAND_CHUNKS=1000 của TOÀN corpus.
# Một doc xếp hạng ~25 thường chỉ có 1-2 chunk lọt top-1000 -> m=3 không bao giờ
# được đáp ứng, và đúng những gold doc bị retriever xếp thấp (ca mà reranker phải
# cứu) lại được cấp ít chunk nhất. Gold doc có p50 = 30 chunk -> reranker đang đọc
# ~10% nội dung.
#
# EMB chỉ ~154k chunk nên full similarity row (q @ EMB.T) đã được dense_search
# tính rồi; lấy top-m trong phạm vi chunk của từng doc là gần như miễn phí.
# CHUNK2DOC tăng dần (chunk sinh theo thứ tự corpus) -> searchsorted cho ngay
# khoảng [lo, hi) chunk của mỗi doc.
import numpy as np, torch

assert np.all(np.diff(CHUNK2DOC) >= 0), \
    'CHUNK2DOC không tăng dần -> searchsorted sai. Chunk phải sinh theo thứ tự corpus.'
_ar = np.arange(len(DOC_IDS))
DOC_CHUNK_LO = np.searchsorted(CHUNK2DOC, _ar, 'left')
DOC_CHUNK_HI = np.searchsorted(CHUNK2DOC, _ar, 'right')
_npd = DOC_CHUNK_HI - DOC_CHUNK_LO
assert _npd.sum() == len(chunks), 'map doc->chunk bị hổng'
print('chunk/doc: min %d  p50 %.0f  mean %.1f  max %d'
      % (_npd.min(), np.percentile(_npd, 50), _npd.mean(), _npd.max()))

# 20 văn bản trong corpus có `passage` RỖNG và cũng không có field `name` (tiêu đề
# chỉ được vá từ slug trong `link` ở Stage 1). chunk_doc() chỉ chunk passage nên
# chúng có 0 chunk -> dense retrieval KHÔNG BAO GIỜ truy hồi được. 6/20 doc này
# từng là gold và 9 câu hỏi train mất trắng vì mọi gold của chúng nằm trong nhóm
# đó. Đây là giới hạn của chunker ở Stage 2, không phải lỗi của cell này: bản B vá
# bằng chunk chỉ-tiêu-đề, còn bản A giữ nguyên chunk set để dùng tiếp cache cũ.
_empty = np.flatnonzero(_npd == 0)
if len(_empty):
    def _unreachable(q):
        gs = [g for g in GOLD[q] if g in DOC2IX]
        return bool(gs) and all(_npd[DOC2IX[g]] == 0 for g in gs)
    _lost = [q for q in GOLD if _unreachable(q)]
    _lost_val = [q for q in _lost if q in set(VAL_QIDS)]
    print(f'{len(_empty)} doc có 0 chunk (passage rỗng) -> không thể truy hồi. '
          f'Ví dụ: {[DOC_IDS[i] for i in _empty[:5]]}')
    print(f'  {len(_lost)} câu train có TOÀN BỘ gold trong nhóm đó '
          f'({len(_lost_val)} câu thuộc val) -> trần recall bị hạ đúng bằng phần này')

def select_chunks(qids, qtext, cand, m, batch=32):
    """-> {qid: {doc_id: [(chunk_ix, dense_score), ...]}}, giảm dần theo điểm.

    Trả kèm dense score để cell fusion dùng làm điểm doc (max chunk) mà không
    phải chạy lại retrieval lần nữa.
    """
    qids, out = list(qids), {}
    for i0 in range(0, len(qtext), batch):
        qv = qmodel.encode(format_dense_queries(qtext[i0:i0 + batch]),
                           batch_size=batch, normalize_embeddings=True,
                           convert_to_tensor=True, show_progress_bar=False).to(EMB_T.dtype)
        S = (qv @ EMB_T.T).float().cpu().numpy()          # (b, n_chunk)
        for r, q in enumerate(qids[i0:i0 + batch]):
            o = {}
            for d in cand[q]:
                lo, hi = DOC_CHUNK_LO[DOC2IX[d]], DOC_CHUNK_HI[DOC2IX[d]]
                if hi <= lo:            # doc passage rỗng -> không có chunk để chấm
                    continue
                loc = S[r, lo:hi]
                k = min(m, hi - lo)
                top = np.argpartition(-loc, k - 1)[:k]
                top = top[np.argsort(-loc[top])]
                o[d] = [(int(lo + j), float(loc[j])) for j in top]
            out[q] = o
    return out

CHUNK_TEXTS = chunks.text.tolist()
assert len(CHUNK_TEXTS) == len(chunks)
print('select_chunks ready | CHUNK_TEXTS =', len(CHUNK_TEXTS))


In [ ]:
#@title Load đúng reranker checkpoint-30 từ Output đã attach
import os, glob, torch
from sentence_transformers import CrossEncoder

matches = []
for root in ART_IN:
    p = f'{root}/models/reranker-ft-{PROFILE}/checkpoint-30'
    if os.path.isfile(f'{p}/model.safetensors'):
        matches.append(p)
assert matches, (
    'Không tìm thấy checkpoint-30. Hãy Add Input -> Your Work -> attach Output '
    'của version có DSC2026_Task1/artifacts/models/reranker-ft-v3b_ft_reranker/checkpoint-30'
)

RERANKER = matches[0]
RERANK_BASE = 'dense'
RR_DEVICES = DEVICES
RERANKERS = {}

def get_reranker(dev):
    if dev not in RERANKERS:
        model = CrossEncoder(RERANKER, max_length=768, device=dev)
        if RERANK_FP16 and dev.startswith('cuda'):
            model.model.half()
        RERANKERS[dev] = model
    return RERANKERS[dev]

for dev in RR_DEVICES:
    get_reranker(dev)
    print('reranker ->', dev, RERANKER)


In [ ]:
#@title 7B.3 — rerank_collect: chấm và lưu điểm CHUNK-LEVEL
import numpy as np, time, threading
from collections import defaultdict

# Lưu điểm từng chunk, KHÔNG pool sẵn -> toàn bộ sweep pooling/fusion ở cell sau
# chạy trên CPU, không tốn thêm một giây GPU nào. Đây là lý do cell này đắt mà
# cell sau miễn phí.
RRC       = {}    # {tag: {qid: {doc_id: [điểm từng chunk]}}}
RRC_DENSE = {}    # {tag: {qid: {doc_id: điểm dense của chunk tốt nhất}}}
RRC_META  = {}    # {tag: (top, m, maxlen)}

def set_rr_maxlen(L):
    """Đổi max_length của mọi reranker đã nạp, không phải tải lại model."""
    for _m in RERANKERS.values():
        _m.max_length = L

def rerank_collect(qids, qtext, cand, top, m, maxlen, tag=None):
    qids = list(qids)
    tag  = tag or f'top{top}_m{m}_L{maxlen}'
    if tag in RRC:
        print(f'[{tag}] đã có trong RRC -> bỏ qua'); return tag
    set_rr_maxlen(maxlen)
    sel = {q: cand[q][:top] for q in qids}

    t0  = time.time()
    rep = select_chunks(qids, qtext, sel, m)
    RRC_DENSE[tag] = {q: {d: v[0][1] for d, v in rep[q].items()} for q in qids}
    print(f'[{tag}] chọn chunk {time.time()-t0:.0f}s | '
          f'{np.mean([len(v) for q in qids for v in rep[q].values()]):.2f} chunk/doc thực nhận',
          flush=True)

    txt   = dict(zip(qids, qtext))
    jobs  = [(q, txt[q], rep[q]) for q in qids]
    parts = [jobs[i::len(RR_DEVICES)] for i in range(len(RR_DEVICES))]
    store, lock, done = {}, threading.Lock(), [0]

    def work(dev, part):
        mdl = get_reranker(dev)
        for q, qt, chunk_of in part:
            pairs, owner = [], []
            for d, cs in chunk_of.items():
                for ci, _ in cs:
                    pairs.append([qt, CHUNK_TEXTS[ci]]); owner.append(d)
            per = defaultdict(list)
            if pairs:
                s = mdl.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
                for d, v in zip(owner, np.asarray(s, dtype=np.float64)):
                    per[d].append(float(v))
            with lock:
                store[q] = dict(per)
                done[0] += 1
                if done[0] % 200 == 0:
                    print(f'  {done[0]}/{len(jobs)} query', flush=True)

    t0  = time.time()
    ths = [threading.Thread(target=work, args=(d, p), daemon=True)
           for d, p in zip(RR_DEVICES, parts) if p]
    for t in ths: t.start()
    for t in ths: t.join()

    RRC[tag], RRC_META[tag] = store, (top, m, maxlen)
    npair = sum(len(v) for q in store for v in store[q].values())
    v = np.array([s for q in store for ss in store[q].values() for s in ss])
    print(f'[{tag}] {npair} cặp | {(time.time()-t0)/60:.1f} phút', flush=True)
    print(f'[{tag}] điểm: min={v.min():.3f} p50={np.median(v):.3f} max={v.max():.3f} '
          f'std={v.std():.3f} | phân biệt {100*len(np.unique(np.round(v,4)))/len(v):.1f}%')
    if not np.isfinite(v).all():
        print('  !! có NaN/Inf -> đặt RERANK_FP16 = False rồi chạy lại')
    if v.std() < 1e-3:
        print('  !! điểm gần như hằng số -> reranker KHÔNG hoạt động, kiểm tra RERANKER')
    return tag

print('ok')


In [ ]:
#@title 7B.4a — pooling + fusion (chỉ định nghĩa hàm)
import numpy as np, pandas as pd

def pool_scores(v, mode='max', tau=0.5, lam=0.3):
    """Gộp điểm các chunk của MỘT doc thành điểm doc.

    'max' là mặc định cũ. Gold doc có p50 = 30 chunk và thường có 2-3 Điều liên
    quan -> 'lse' (max mềm) và 'top2' thưởng thêm cho doc có nhiều chunk khớp,
    trong khi vẫn không phạt doc dài chỉ có 1 Điều đúng như 'mean' làm.
    """
    v = np.asarray(v, float)
    if len(v) == 0:
        return -1e30
    if mode == 'max':
        return float(v.max())
    if mode == 'mean':
        return float(v.mean())
    if mode == 'top2':
        return float(v.max() + lam * (np.sort(v)[-2] if len(v) > 1 else v.max()))
    if mode == 'lse':
        mx = float(v.max())
        return mx + tau * float(np.log(np.exp((v - mx) / tau).sum()))
    raise ValueError(mode)

def _z(d):
    """z-norm trong PHẠM VI MỘT CÂU. Điểm cross-encoder không cùng thang giữa các
    câu hỏi, chuẩn hoá toàn cục là trộn hai phân bố khác nhau."""
    v = np.array(list(d.values()), float)
    mu, sd = v.mean(), v.std() + 1e-9
    return {k: (x - mu) / sd for k, x in d.items()}

def rank_from(tag, cand, pool='max', tau=0.5, lam=0.3, fuse='score',
              alpha=0.0, wc=1.0, wr=1.0, k=RRF_K):
    """-> {qid: [doc_id đã xếp hạng]}.

    fuse='score': z(rerank) + alpha*z(dense) — giữ được MARGIN điểm, thứ mà RRF
                  vứt bỏ. Với top-5 legal IR, margin là thông tin thật.
    fuse='rrf'  : RRF giữa thứ tự retriever và thứ tự rerank. Bản cũ ghim w=(1,1)
                  — đúng sai lầm mà Stage 6 đã học được với BM25.
    """
    rr, dn, out = RRC[tag], RRC_DENSE[tag], {}
    for q in rr:
        sel = [d for d in cand[q] if d in rr[q]]
        if not sel:
            out[q] = list(cand[q]); continue
        pooled = {d: pool_scores(rr[q][d], pool, tau, lam) for d in sel}
        if fuse == 'score':
            zr = _z(pooled)
            zd = _z({d: dn[q].get(d, 0.0) for d in sel}) if alpha else None
            key = (lambda d: -(zr[d] + alpha * zd[d])) if alpha else (lambda d: -zr[d])
            out[q] = sorted(sel, key=key) + [d for d in cand[q] if d not in rr[q]]
        else:
            s = {}
            for i, d in enumerate(cand[q]):
                s[d] = s.get(d, 0.0) + wc / (k + i + 1)
            for i, d in enumerate(sorted(sel, key=lambda d: -pooled[d])):
                s[d] = s.get(d, 0.0) + wr / (k + i + 1)
            out[q] = [d for d, _ in sorted(s.items(), key=lambda x: -x[1])]
    return out

print('ok — rank_from(tag, cand, pool=..., fuse=..., ...)')


In [ ]:
#@title Cấu hình tốt nhất đã đo trên validation
BEST = {
    'tag': 'ft_top30_m8_L768',
    'pool': 'top2', 'tau': 0.5, 'lam': 0.2,
    'fuse': 'score', 'alpha': 0.8,
    'wc': 1.0, 'wr': 1.0,
    'top': 30, 'm': 8, 'maxlen': 768,
}
print('BEST =', BEST)


In [ ]:
#@title Kiểm chứng lại Recall@5 = 0.9524 trên validation
# Cell này tốn GPU gần tương đương public inference. Đặt False nếu đã kiểm chứng xong.
RUN_VALIDATION_CHECK = True

if RUN_VALIDATION_CHECK:
    val_qids_check = list(VAL_QIDS)
    val_text_check = [QUESTIONS[q] for q in val_qids_check]
    val_cand_check = retrieve_dense(val_qids_check, val_text_check)
    val_tag_check = 'verify_ft_top30_m8_L768'

    rerank_collect(
        val_qids_check, val_text_check, val_cand_check,
        BEST['top'], BEST['m'], BEST['maxlen'], tag=val_tag_check
    )
    val_preds_check = rank_from(
        val_tag_check, val_cand_check,
        BEST['pool'], BEST['tau'], BEST['lam'],
        BEST['fuse'], BEST['alpha'], BEST['wc'], BEST['wr']
    )

    gold_val_check = {q: GOLD[q] for q in val_qids_check}
    metrics_check = evaluate(val_preds_check, gold_val_check)
    print('VALIDATION CHECK =', metrics_check)
    print('Recall@5          = %.6f' % metrics_check['recall@5'])
    print('OFFICIAL_recall   = %.6f' % metrics_check['OFFICIAL_recall'])
    print('OFFICIAL_precision=', '%.6f' % metrics_check['OFFICIAL_precision'])

    expected = 0.9524
    actual = metrics_check['OFFICIAL_recall']
    assert abs(actual - expected) <= 0.001, (
        f'Recall đo lại {actual:.6f} lệch khỏi {expected:.4f}; '
        'không chạy public trước khi kiểm tra model/config.'
    )
    print('OK: tái lập đúng mức Recall 0.9524 (sai số <= 0.001).')
else:
    print('Bỏ qua validation check.')


In [ ]:
#@title Chạy đúng BEST trên public test
import time

def full_pipeline_best(qids, qtext):
    qids = list(qids)
    cand = retrieve_dense(qids, qtext)
    tag = 'pub_' + BEST['tag']
    rerank_collect(qids, qtext, cand, BEST['top'], BEST['m'], BEST['maxlen'], tag=tag)
    return rank_from(tag, cand, BEST['pool'], BEST['tau'], BEST['lam'],
                     BEST['fuse'], BEST['alpha'], BEST['wc'], BEST['wr'])

pub_qids = list(PUBLIC.keys())
pub_text = [PUBLIC[q] for q in pub_qids]
print("checkpoint-30 | dense | top30 x 8 chunks | L768 | top2 | score alpha=0.8")
t0 = time.time()
pub_preds = full_pipeline_best(pub_qids, pub_text)
assert len(pub_preds) == len(pub_qids), (len(pub_preds), len(pub_qids))
print('%d câu | %.1f phút' % (len(pub_qids), (time.time() - t0) / 60))


In [ ]:
#@title Validate + zip
import json, zipfile, os, collections

def build_submission(preds, all_qids, corpus_ids):
    """Luôn trả về ĐÚNG 5 id/câu. Thiếu thì bù bằng ứng viên tốt nhất còn lại."""
    sub, filler = {}, [d for d in DOC_IDS[:TOPK_SUBMIT]]
    for q in all_qids:
        ids, seen = [], set()
        for d in preds.get(q, []):
            d = str(d)
            if d not in seen and d in corpus_ids:
                seen.add(d); ids.append(d)
            if len(ids) == TOPK_SUBMIT:
                break
        for f in filler:                       # câu không có ứng viên -> vẫn đủ 5
            if len(ids) == TOPK_SUBMIT:
                break
            if f not in seen:
                seen.add(f); ids.append(f)
        sub[q] = {'answer': ids}
    return sub

def validate(sub, all_qids, corpus_ids):
    errs = []
    # scoring.py của BTC: len(ids_preds) != len(ids_truth) -> raise Exception
    # -> lệch số câu làm SẬP TOÀN BỘ bài nộp, không phải chỉ mất điểm 1 câu.
    if len(sub) != len(all_qids):
        errs.append(f'SẬP TOÀN BÀI: {len(sub)} câu nộp vs {len(all_qids)} câu đề')
    if set(sub) != set(all_qids):
        errs.append(f'SẬP TOÀN BÀI: thiếu {len(set(all_qids)-set(sub))} qid, '
                    f'thừa {len(set(sub)-set(all_qids))} qid (len(None) -> TypeError)')
    for q, v in sub.items():
        a = v.get('answer')
        if not isinstance(a, list):
            errs.append(f'{q}: answer không phải list'); continue
        if len(a) == 0:
            errs.append(f'{q}: answer rỗng -> câu này 0 điểm')
        if len(a) > TOPK_SUBMIT:
            errs.append(f'{q}: {len(a)} id > {TOPK_SUBMIT} -> CÂU NÀY BỊ 0 ĐIỂM')
        if len(a) < TOPK_SUBMIT:
            errs.append(f'{q}: chỉ {len(a)} id -> mất điểm Recall vô cớ')
        if len(set(a)) != len(a):
            errs.append(f'{q}: có id trùng lặp')
        if any(not isinstance(x, str) for x in a):
            errs.append(f'{q}: id phải là string')
        if any(x not in corpus_ids for x in a):
            errs.append(f'{q}: có id không tồn tại trong corpus')
    return errs

corpus_ids = set(corpus.doc_id)
sub = build_submission(pub_preds, list(PUBLIC), corpus_ids)
errs = validate(sub, list(PUBLIC), corpus_ids)

if errs:
    print(f'{len(errs)} LỖI — chưa nộp được:')
    for e in errs[:20]:
        print('  -', e)
else:
    # Kaggle: ghi thẳng vào GỐC /kaggle/working -> hiện ngay tab Output, tải 1 click.
    # KHÔNG ghi vào /kaggle/temp: thư mục đó không vào output, session chết là mất file.
    SUB_DIR  = '/kaggle/working' if ON_KAGGLE else ART
    out_json = f'{SUB_DIR}/submission.json'
    out_zip  = f'{SUB_DIR}/submission_{PROFILE}.zip'
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(sub, f, ensure_ascii=False)
    with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
        z.write(out_json, arcname='submission.json')   # arcname -> nằm ở GỐC zip
    with zipfile.ZipFile(out_zip) as z:
        assert z.namelist() == ['submission.json'], z.namelist()
    print('OK ->', out_zip, '(%.0f KB)' % (os.path.getsize(out_zip) / 1e3))
    print('     ', out_json)
    if ON_KAGGLE:
        print('Tải về: panel bên phải -> Output -> /kaggle/working -> '
              f'submission_{PROFILE}.zip (hoặc Save Version rồi tải từ tab Output).')
        print('Nộp file ZIP lên Codabench, không nộp file .json.')
    print('phân bố số id/câu:', collections.Counter(len(v['answer']) for v in sub.values()))
    print(json.dumps(dict(list(sub.items())[:2]), ensure_ascii=False, indent=1))

## Kết quả

Nếu validator báo `OK`, tải `/kaggle/working/submission_v3b_ft_reranker.zip` và nộp ZIP.
Không cần Save & Run All để tải file trong session đang mở.
